# Robustness Sweeps Across Noise and Subsampling

This notebook explores how the shipped `v0.6` stack behaves under simple robustness perturbations.

It uses:

- `add_gaussian_noise(...)`
- `subsample_time(...)`
- `subsample_x(...)`
- `split_batch_train_heldout(...)`
- `evaluate_discovery_recovery(...)`
- `summarize_recovery_grid(...)`

The recovery metric here is **not** PDE-truth recovery. It compares backend-native sparse terms from each perturbed fit against a clean baseline backend-native fit on the same primary state feature.


In [ ]:
import importlib.util

if importlib.util.find_spec("pysindy") is None:
    raise RuntimeError("Install pdelie[downstream] or pdelie[test] to run this notebook.")

import numpy as np

from pdelie import GeneratorFamily
from pdelie.data import (
    add_gaussian_noise,
    generate_heat_1d_field_batch,
    split_batch_train_heldout,
    subsample_time,
    subsample_x,
)
from pdelie.discovery import (
    build_translation_canonical_discovery_inputs,
    evaluate_discovery_recovery,
    fit_pysindy_discovery,
    summarize_recovery_grid,
    to_pysindy_trajectories,
)


In [ ]:
def make_known_translation_generator() -> GeneratorFamily:
    basis_spec = {
        "variables": ["t", "x", "u"],
        "component_names": ["xi"],
        "basis_terms": [
            {"label": "1", "powers": [0, 0, 0]},
            {"label": "t", "powers": [1, 0, 0]},
            {"label": "x", "powers": [0, 1, 0]},
            {"label": "u", "powers": [0, 0, 1]},
        ],
        "component_ordering": ["xi"],
        "term_ordering": ["1", "t", "x", "u"],
        "layout": "component_major",
    }
    return GeneratorFamily(
        parameterization="polynomial_translation_affine",
        coefficients=np.array([[1.0, 0.0, 0.0, 0.0]], dtype=float),
        basis_spec=basis_spec,
        normalization="l2_unit",
        diagnostics={},
    )


def extract_primary_terms(fit_result, feature_name):
    return dict(fit_result["equation_terms"].get(feature_name, {}))


def initial_batch_std_by_feature(trajectories) -> np.ndarray:
    stacked = np.stack(trajectories, axis=0)
    return np.std(stacked[:, 0, :], axis=0)


def choose_inspected_feature(raw_trajectories, canonical_trajectories, feature_names):
    raw_std = initial_batch_std_by_feature(raw_trajectories)
    canonical_std = initial_batch_std_by_feature(canonical_trajectories)
    reduction = raw_std - canonical_std
    best_index = int(np.argmax(reduction))
    return feature_names[best_index]


def feature_std_summary(raw_trajectories, canonical_trajectories, feature_names, feature_name):
    raw_std = initial_batch_std_by_feature(raw_trajectories)
    canonical_std = initial_batch_std_by_feature(canonical_trajectories)
    feature_index = feature_names.index(feature_name)
    return (
        float(raw_std[feature_index]),
        float(canonical_std[feature_index]),
        float(raw_std[feature_index] - canonical_std[feature_index]),
    )


def run_condition(base_field, generator, *, std_fraction, noise_seed, time_stride, x_stride):
    noisy = add_gaussian_noise(base_field, std_fraction=std_fraction, seed=noise_seed)
    time_subsampled = subsample_time(noisy, stride=time_stride)
    x_subsampled = subsample_x(time_subsampled, stride=x_stride)
    train_field, heldout_field = split_batch_train_heldout(x_subsampled, train_size=2, seed=7000)

    raw_trajectories, raw_time_values, raw_feature_names = to_pysindy_trajectories(train_field)
    discovery_inputs = build_translation_canonical_discovery_inputs(train_field, generator_family=generator)
    fit_result = fit_pysindy_discovery(
        discovery_inputs["trajectories"],
        discovery_inputs["time_values"],
        discovery_inputs["feature_names"],
    )
    primary_feature = choose_inspected_feature(
        raw_trajectories,
        discovery_inputs["trajectories"],
        raw_feature_names,
    )
    raw_initial_batch_std, canonical_initial_batch_std, initial_batch_std_reduction = feature_std_summary(
        raw_trajectories,
        discovery_inputs["trajectories"],
        raw_feature_names,
        primary_feature,
    )
    return {
        "train_field": train_field,
        "heldout_field": heldout_field,
        "raw_trajectories": raw_trajectories,
        "raw_time_values": raw_time_values,
        "raw_feature_names": raw_feature_names,
        "inputs": discovery_inputs,
        "fit": fit_result,
        "primary_feature": primary_feature,
        "raw_initial_batch_std": raw_initial_batch_std,
        "canonical_initial_batch_std": canonical_initial_batch_std,
        "initial_batch_std_reduction": initial_batch_std_reduction,
        "primary_terms": extract_primary_terms(fit_result, primary_feature),
    }


In [ ]:
base_field = generate_heat_1d_field_batch(batch_size=4, num_times=33, num_points=64, seed=200)
generator = make_known_translation_generator()

baseline_run = run_condition(
    base_field,
    generator,
    std_fraction=0.0,
    noise_seed=9000,
    time_stride=1,
    x_stride=1,
)
baseline_terms = baseline_run["primary_terms"]
baseline_feature = baseline_run["primary_feature"]

print("baseline inspected feature:", baseline_feature)
print("baseline fit status:", baseline_run["fit"]["status"])
print("baseline nonzero backend terms:", len(baseline_terms))
print("baseline raw initial-time batch std:", baseline_run["raw_initial_batch_std"])
print("baseline canonical initial-time batch std:", baseline_run["canonical_initial_batch_std"])
print("baseline initial-time batch std reduction:", baseline_run["initial_batch_std_reduction"])


## Sweep conditions

We repeat each condition across a few noise seeds so `summarize_recovery_grid(...)` has multiple records per condition.
When sparse backend terms are empty, the inspected feature and initial-time batch-variability reduction still provide a useful structural signal.


In [ ]:
records = []
fit_rows = []

for std_fraction in [0.0, 1e-3, 1e-2, 5e-2]:
    for time_stride in [1, 2, 4]:
        for x_stride in [1, 2, 4]:
            for noise_seed in [9100, 9101, 9102]:
                run = run_condition(
                    base_field,
                    generator,
                    std_fraction=std_fraction,
                    noise_seed=noise_seed,
                    time_stride=time_stride,
                    x_stride=x_stride,
                )
                recovery = evaluate_discovery_recovery(baseline_terms, run["primary_terms"])
                records.append(
                    {
                        "conditions": {
                            "noise_std_fraction": std_fraction,
                            "time_stride": time_stride,
                            "x_stride": x_stride,
                        },
                        "recovery": recovery,
                    }
                )
                fit_rows.append(
                    {
                        "noise_std_fraction": std_fraction,
                        "time_stride": time_stride,
                        "x_stride": x_stride,
                        "noise_seed": noise_seed,
                        "fit_status": run["fit"]["status"],
                        "inspected_feature": run["primary_feature"],
                        "primary_nonzero_terms": len(run["primary_terms"]),
                        "raw_initial_batch_std": run["raw_initial_batch_std"],
                        "canonical_initial_batch_std": run["canonical_initial_batch_std"],
                        "initial_batch_std_reduction": run["initial_batch_std_reduction"],
                        "classification": recovery["classification"],
                    }
                )

fit_rows[:5]


In [ ]:
summary_rows = summarize_recovery_grid(records)
summary_rows[:8]


In [ ]:
structural_rows = []
grouped_fit_rows = {}
for row in fit_rows:
    key = (row["noise_std_fraction"], row["time_stride"], row["x_stride"])
    grouped_fit_rows.setdefault(key, []).append(row)

for (noise_std_fraction, time_stride, x_stride), rows_for_condition in sorted(grouped_fit_rows.items()):
    structural_rows.append(
        {
            "conditions": {
                "noise_std_fraction": noise_std_fraction,
                "time_stride": time_stride,
                "x_stride": x_stride,
            },
            "mean_primary_nonzero_terms": float(np.mean([row["primary_nonzero_terms"] for row in rows_for_condition])),
            "mean_raw_initial_batch_std": float(np.mean([row["raw_initial_batch_std"] for row in rows_for_condition])),
            "mean_canonical_initial_batch_std": float(np.mean([row["canonical_initial_batch_std"] for row in rows_for_condition])),
            "mean_initial_batch_std_reduction": float(np.mean([row["initial_batch_std_reduction"] for row in rows_for_condition])),
        }
    )

structural_rows[:8]


## A few easy comparisons

These are simple slices through the grouped summaries. They are often enough to spot where the current `v0.6` stack begins to look structurally brittle.
When recovery against the sparse backend baseline is trivial, the structural variability-reduction summary is the more informative readout.


In [ ]:
exact_rows = [row for row in summary_rows if row["exact_count"] > 0]
failed_rows = [row for row in summary_rows if row["failed_count"] > 0]

print("num grouped conditions:", len(summary_rows))
print("conditions with at least one exact recovery against baseline support:", len(exact_rows))
print("conditions with at least one failed recovery against baseline support:", len(failed_rows))

print("\nRepresentative grouped rows:")
for row in summary_rows[:5]:
    print(row)


In [ ]:
best_structural_rows = sorted(
    structural_rows,
    key=lambda row: row["mean_initial_batch_std_reduction"],
    reverse=True,
)

print("Top conditions by mean initial-time batch std reduction:")
for row in best_structural_rows[:5]:
    print(row)
